> **Note:** This notebook applies the same pipeline as `molecular_property_prediction_dopamine.ipynb` but uses the **MolNet Lipophilicity dataset (~4,200 molecules)** instead of the dopamine receptor dataset (479 molecules). The larger dataset allows for more robust evaluation of the same multi-modal representation fusion approach.

# Data preprocessing

Loading the MolNet Lipophilicity dataset which contains 4200 molecules.

Also, this exists:

metric = dc.metrics.Metric(dc.metrics.mean_absolute_error)

model.evaluate(test_dataset, [metric], transformers)

In [1]:
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_lipo(
    featurizer='ecfp',  # use ECFP featurizer
    splitter='scaffold',
    reload=False  # disable loading from cached directory
)


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [2]:
(train, val, test) = datasets
# train.ids = smiles; train.y = labels

len(train), len(val), len(test)

(3360, 420, 420)

In [3]:
# canonicalization of SMILES
from rdkit import Chem
import numpy as np

def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

def assert_valid_smiles(smiles_list, split_name):
    bad = [i for i, s in enumerate(smiles_list) if s is None]
    if bad:
        raise ValueError(
            f"{split_name} has {len(bad)} invalid SMILES after canonicalization; "
            "cannot proceed without dropping molecules."
        )

train_smiles = [canonicalize(smi) for smi in train.ids]
val_smiles = [canonicalize(smi) for smi in val.ids]
test_smiles = [canonicalize(smi) for smi in test.ids]

assert_valid_smiles(train_smiles, "train")
assert_valid_smiles(val_smiles, "val")
assert_valid_smiles(test_smiles, "test")

train_y = train.y.reshape(-1)
val_y   = val.y.reshape(-1)
test_y  = test.y.reshape(-1)


In [4]:
# checking whether there is any missing data

import pandas as pd

data_train = pd.DataFrame({'smiles': train_smiles, 'target': train_y.astype(float)})
data_val = pd.DataFrame({'smiles': val_smiles, 'target': val_y.astype(float)})
data_test = pd.DataFrame({'smiles': test_smiles, 'target': test_y.astype(float)})

data = pd.concat([data_train, data_val, data_test], ignore_index=True)

data.isna().sum()

def untransform_y(y, transformers):
    y_u = np.asarray(y)
    if transformers is None:
        return y_u
    for t in transformers:
        if getattr(t, "transform_y", False):
            y_u = t.untransform(y_u.reshape(-1, 1)).ravel()
    return y_u

smiles_all = np.concatenate([
    data_train["smiles"].values,
    data_val["smiles"].values,
    data_test["smiles"].values
])
y_all = np.concatenate([
    data_train["target"].values,
    data_val["target"].values,
    data_test["target"].values
])


smiles    0
target    0
dtype: int64

No need to manually scale, that is done while loading the dataset with DeepChem. 

It is important to untransform predictions for reporting.

DeepChem also already handles outliers: standardizes targets using meand & std, does not assume Gaussian tails, leaves the distribution intact

In [5]:
from rdkit.Chem.Scaffolds import MurckoScaffold
import numpy as np
from sklearn.model_selection import GroupKFold

def murcko_scaffold(smiles_list):
    scaffolds = []
    for s in smiles_list:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(smiles=s, includeChirality=False)
        scaffolds.append(scaffold)
    return np.array(scaffolds)

# Molecular representations

In [6]:
# !pip install rdkit-pypi mordred pandas numpy

## Molecular Descriptors

In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from mordred import Calculator, descriptors


# function to clean mordred dataframe (no row/column dropping)
def clean_mordred_df(df):
    X = df.drop(columns=["smiles", "target"]).copy()

    # convert to numeric; invalid parsing -> NaN
    X = X.apply(pd.to_numeric, errors='coerce')

    # replace inf with NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    clean_df = pd.concat([df[["smiles", "target"]], X], axis=1)
    return clean_df


# function to compute 2D Mordred descriptors (keeps all rows)
def compute_mordred_2d(df):
    mols = []
    valid_idx = []
    bad_smiles = []

    for idx, row in df.iterrows():
        s = row["smiles"]
        m = Chem.MolFromSmiles(s)
        if m is None:
            bad_smiles.append(s)
        else:
            mols.append(m)
            valid_idx.append(idx)
    print("Valid:", len(valid_idx), "Invalid:", len(bad_smiles))

    calc = Calculator(descriptors, ignore_3D=True)
    mordred_valid = calc.pandas(mols)
    mordred_valid.index = valid_idx

    # keep row count aligned with input split
    mordred_df = pd.DataFrame(index=df.index, columns=mordred_valid.columns, dtype=float)
    mordred_df.loc[valid_idx] = mordred_valid

    mordred_df.insert(0, "smiles", df["smiles"].values)
    mordred_df.insert(1, "target", df["target"].values)

    clean_df = clean_mordred_df(mordred_df)
    print("Number of descriptors:", clean_df.shape[1])

    return clean_df


# function to compute 3D Mordred descriptors (keeps all rows)
def compute_mordred_3d(df):
    mols = []
    valid_idx = []
    bad_smiles = []

    params = AllChem.ETKDGv3()
    params.randomSeed = 0  # for reproducibility

    for idx, row in df.iterrows():
        s = row["smiles"]
        m = Chem.MolFromSmiles(s)
        if m is None:
            bad_smiles.append(s)
            continue

        m_3d = Chem.AddHs(m)
        coded = AllChem.EmbedMolecule(m_3d, params)
        ok = coded == 0
        if ok:
            try:
                if AllChem.MMFFHasAllMoleculeParams(m_3d):
                    AllChem.MMFFOptimizeMolecule(m_3d)
                else:
                    AllChem.UFFOptimizeMolecule(m_3d)
            except Exception as e:
                ok = False
                print("Optimize failed for SMILES:", s, "Error:", e)

        if ok:
            mols.append(m_3d)
            valid_idx.append(idx)
        else:
            bad_smiles.append(s)

    print("Valid:", len(valid_idx), "Invalid:", len(bad_smiles))

    calc3d = Calculator(descriptors, ignore_3D=False)
    mordred_valid = calc3d.pandas(mols)
    mordred_valid.index = valid_idx

    mordred_df = pd.DataFrame(index=df.index, columns=mordred_valid.columns, dtype=float)
    mordred_df.loc[valid_idx] = mordred_valid

    mordred_df.insert(0, "smiles", df["smiles"].values)
    mordred_df.insert(1, "target", df["target"].values)

    clean_df = clean_mordred_df(mordred_df)
    print("Number of descriptors:", clean_df.shape[1])

    return clean_df


For now, I will only use 2d descriptors since the computation of 3D descriptors takes a substantial amount of time. I will compute 3D descriptors later and validate them on the validation set to choose a better fit.

Generation of 2D molecular descriptors takes around 1.30 minutes. Generation od 3D molecular descriptors takes around 8 minutes.

In [8]:
from pathlib import Path

def load_or_compute_mordred(cache_path, df, compute_fn, recompute=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not recompute:
        df_desc = pd.read_pickle(cache_path)
    else:
        if compute_fn is None:
            raise ValueError(f"Cache missing and no compute_fn provided: {cache_path}")
        df_desc = compute_fn(df)
        df_desc.to_pickle(cache_path)

    # audit: preserve split size and order for MolNet comparability
    if len(df_desc) != len(df):
        raise ValueError(
            f"{cache_path} row count mismatch ({len(df_desc)} vs {len(df)}). "
            "Recompute descriptors without dropping rows."
        )
    if not np.array_equal(df_desc["smiles"].values, df["smiles"].values):
        raise ValueError(
            f"{cache_path} SMILES order mismatch. "
            "Recompute descriptors from the canonicalized split."
        )
    return df_desc


RECOMPUTE_2D = False  # set True to rebuild caches with no row-dropping
mordred_df_2d_train = load_or_compute_mordred("./cache/mordred_2d_lipo_train.pkl", data_train, compute_mordred_2d, recompute=RECOMPUTE_2D)
mordred_df_2d_val = load_or_compute_mordred("./cache/mordred_2d_lipo_val.pkl", data_val, compute_mordred_2d, recompute=RECOMPUTE_2D)
mordred_df_2d_test = load_or_compute_mordred("./cache/mordred_2d_lipo_test.pkl", data_test, compute_mordred_2d, recompute=RECOMPUTE_2D)

if not mordred_df_2d_train.columns.equals(mordred_df_2d_val.columns) or not mordred_df_2d_train.columns.equals(mordred_df_2d_test.columns):
    raise ValueError("2D descriptor columns differ across splits; recompute caches without per-split filtering.")


In [9]:
RECOMPUTE_3D = False  # set True to rebuild caches (3D is slower)
mordred_df_3d_train = load_or_compute_mordred("./cache/mordred_3d_lipo_train.pkl", data_train, compute_mordred_3d, recompute=RECOMPUTE_3D)
mordred_df_3d_val = load_or_compute_mordred("./cache/mordred_3d_lipo_val.pkl", data_val, compute_mordred_3d, recompute=RECOMPUTE_3D)
mordred_df_3d_test = load_or_compute_mordred("./cache/mordred_3d_lipo_test.pkl", data_test, compute_mordred_3d, recompute=RECOMPUTE_3D)

if not mordred_df_3d_train.columns.equals(mordred_df_3d_val.columns) or not mordred_df_3d_train.columns.equals(mordred_df_3d_test.columns):
    raise ValueError("3D descriptor columns differ across splits; recompute caches without per-split filtering.")


### Scaffolding functions

In [10]:
import numpy as np
import pandas as pd

from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.feature_selection import VarianceThreshold, RFECV
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

def murcko_scaffold(smiles: str) -> str:
    return MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles, includeChirality=False)

def clean_descriptor_matrix(X: pd.DataFrame) -> pd.DataFrame:
    Xc = X.apply(pd.to_numeric, errors="coerce")
    Xc = Xc.replace([np.inf, -np.inf], np.nan)
    return Xc

def prepare_descriptor_matrices(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    *,
    max_missing_frac=0.2,
    impute_strategy="median"
):
    Xtr = clean_descriptor_matrix(X_train)
    Xva = clean_descriptor_matrix(X_val)
    Xte = clean_descriptor_matrix(X_test)

    keep_cols = Xtr.columns[Xtr.isna().mean() <= max_missing_frac].tolist()
    if not keep_cols:
        raise ValueError("No descriptor columns left after missingness filtering.")

    Xtr = Xtr[keep_cols]
    Xva = Xva.reindex(columns=keep_cols)
    Xte = Xte.reindex(columns=keep_cols)

    imputer = SimpleImputer(strategy=impute_strategy)
    Xtr = pd.DataFrame(imputer.fit_transform(Xtr), columns=keep_cols, index=Xtr.index)
    Xva = pd.DataFrame(imputer.transform(Xva), columns=keep_cols, index=Xva.index)
    Xte = pd.DataFrame(imputer.transform(Xte), columns=keep_cols, index=Xte.index)

    return Xtr, Xva, Xte, keep_cols

def drop_autocorr_cols(X: pd.DataFrame, keywords=("ATSC", "AATS")) -> pd.DataFrame:
    mask = X.columns.str.contains("|".join(keywords))
    return X.loc[:, ~mask]

def apply_variance_threshold_train(Xtr: pd.DataFrame, Xva: pd.DataFrame, Xte: pd.DataFrame,
                                   threshold: float = 1e-8):
    vt = VarianceThreshold(threshold=threshold)
    vt.fit(Xtr)
    kept = Xtr.columns[vt.get_support()].tolist()
    return Xtr[kept], Xva[kept], Xte[kept], kept

def prune_correlated_features_by_importance(X: pd.DataFrame, y,
                                           threshold: float = 0.90,
                                           random_state: int = 42,
                                           n_jobs: int = -1):
    # Fit a quick RF to get importances; drop one from each highly-correlated pair.
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=2,
        n_jobs=n_jobs,
        random_state=random_state
    )
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=X.columns)

    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        high = upper.index[upper[col] > threshold].tolist()
        for row in high:
            drop = row if imp[row] < imp[col] else col
            to_drop.add(drop)

    kept = [c for c in X.columns if c not in to_drop]
    return X[kept], kept


In [11]:
def fit_fast_selector_and_transform(
    X_train: pd.DataFrame, y_train, groups_train,
    X_val: pd.DataFrame, X_test: pd.DataFrame,
    *,
    autocorr_keywords=("ATSC", "AATS"),
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1,
    max_missing_frac=0.2,
    impute_strategy="median",
    random_state=42,
    n_jobs=-1
):
    # 0) missingness filter + impute using training only
    Xtr, Xva, Xte, kept_missing = prepare_descriptor_matrices(
        X_train, X_val, X_test,
        max_missing_frac=max_missing_frac,
        impute_strategy=impute_strategy
    )

    # 1) drop autocorr
    Xtr = drop_autocorr_cols(Xtr, keywords=autocorr_keywords)
    Xva = drop_autocorr_cols(Xva, keywords=autocorr_keywords)
    Xte = drop_autocorr_cols(Xte, keywords=autocorr_keywords)

    # 2) VT
    Xtr, Xva, Xte, kept_vt = apply_variance_threshold_train(Xtr, Xva, Xte, threshold=vt_threshold)

    # 3) correlation prune (train-only)
    Xtr, kept_corr = prune_correlated_features_by_importance(
        Xtr, y_train, threshold=corr_threshold, random_state=random_state, n_jobs=n_jobs
    )
    Xva = Xva[kept_corr]
    Xte = Xte[kept_corr]

    # 4) top-k by quick RF importance (train-only)
    rf_rank = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        n_jobs=n_jobs,
        random_state=random_state
    )
    rf_rank.fit(Xtr, y_train)
    imp = pd.Series(rf_rank.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
    top_k = min(top_k, len(imp))
    kept_topk = imp.head(top_k).index.tolist()

    Xtr_k = Xtr[kept_topk]
    Xva_k = Xva[kept_topk]
    Xte_k = Xte[kept_topk]

    # 5) light RFECV on the reduced set (scaffold-aware)
    base_rf = RandomForestRegressor(
        n_estimators=200,          # keep light inside RFECV
        max_depth=12,
        min_samples_leaf=2,
        n_jobs=n_jobs,
        random_state=random_state
    )
    cv = GroupKFold(n_splits=rfecv_cv_splits)
    selector = RFECV(
        estimator=base_rf,
        step=rfecv_step,           # 0.1 = drop 10% per iteration
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=n_jobs,
        min_features_to_select=min_features,
        verbose=0
    )
    selector.fit(Xtr_k, y_train, groups=groups_train)
    kept_final = Xtr_k.columns[selector.support_].tolist()

    info = {
        "kept_missing": kept_missing,
        "kept_vt": kept_vt,
        "kept_corr": kept_corr,
        "kept_topk": kept_topk,
        "kept_final": kept_final
    }

    return Xtr_k[kept_final], Xva_k[kept_final], Xte_k[kept_final], info, selector


In [12]:
from sklearn.model_selection import GroupKFold

def scaffold_cv_fast_selector_rf(
    X_all: pd.DataFrame,
    y_all,
    smiles_all,
    *,
    n_splits=5,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=4,
    rfecv_step=0.1,
    max_missing_frac=0.2,
    impute_strategy="median",
    random_state=42,
    n_jobs=-1
):
    groups_all = np.array([murcko_scaffold(s) for s in smiles_all])
    outer = GroupKFold(n_splits=n_splits)

    maes, rmses, nfeats = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(outer.split(X_all, y_all, groups_all), 1):
        Xtr, ytr = X_all.iloc[tr_idx], np.asarray(y_all)[tr_idx]
        Xva, yva = X_all.iloc[va_idx], np.asarray(y_all)[va_idx]
        groups_tr = groups_all[tr_idx]

        Xtr_sel, Xva_sel, _, info, _ = fit_fast_selector_and_transform(
            Xtr, ytr, groups_tr,
            Xva, Xva,  # dummy test
            vt_threshold=vt_threshold,
            corr_threshold=corr_threshold,
            top_k=top_k,
            min_features=min_features,
            rfecv_cv_splits=rfecv_cv_splits,
            rfecv_step=rfecv_step,
            max_missing_frac=max_missing_frac,
            impute_strategy=impute_strategy,
            random_state=random_state,
            n_jobs=n_jobs
        )

        model = RandomForestRegressor(
            n_estimators=1500,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=n_jobs,
            random_state=random_state
        )
        model.fit(Xtr_sel, ytr)
        pred = model.predict(Xva_sel)

        yva_u = untransform_y(yva, transformers)
        pred_u = untransform_y(pred, transformers)
        mae = mean_absolute_error(yva_u, pred_u)
        rmse = mean_squared_error(yva_u, pred_u, squared=False)

        maes.append(mae)
        rmses.append(rmse)
        nfeats.append(len(info["kept_final"]))

        print(f"Fold {fold}: MAE={mae:.4f} RMSE={rmse:.4f} n_feat={nfeats[-1]}")

    print("5-fold scaffold CV:")
    print(f"MAE  mean+/-std: {np.mean(maes):.4f} +/- {np.std(maes):.4f}")
    print(f"RMSE mean+/-std: {np.mean(rmses):.4f} +/- {np.std(rmses):.4f}")
    print(f"n_feat mean+/-std: {np.mean(nfeats):.1f} +/- {np.std(nfeats):.1f}")

    return maes, rmses, nfeats


In [13]:
# Official MolNet benchmark (no leakage: train-only preprocessing)
X_desc_train = mordred_df_2d_train.drop(columns=["smiles", "target"])
y_train = mordred_df_2d_train["target"].values
smiles_train = mordred_df_2d_train["smiles"].values

X_desc_val = mordred_df_2d_val.drop(columns=["smiles", "target"])
y_val = mordred_df_2d_val["target"].values

X_desc_test = mordred_df_2d_test.drop(columns=["smiles", "target"])
y_test = mordred_df_2d_test["target"].values

groups_train = np.array([murcko_scaffold(s) for s in smiles_train])

Xtr_sel, Xva_sel, Xte_sel, info, rfecv_obj = fit_fast_selector_and_transform(
    X_desc_train, y_train, groups_train,
    X_desc_val, X_desc_test,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1
)

print("Selected features:", len(info["kept_final"]))

final_rf = RandomForestRegressor(
    n_estimators=1500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
final_rf.fit(Xtr_sel, y_train)

pred_val = final_rf.predict(Xva_sel)
pred_test = final_rf.predict(Xte_sel)

y_val_u = untransform_y(y_val, transformers)
pred_val_u = untransform_y(pred_val, transformers)
y_test_u = untransform_y(y_test, transformers)
pred_test_u = untransform_y(pred_test, transformers)

print("Official VAL  MAE/RMSE:",
      mean_absolute_error(y_val_u, pred_val_u),
      mean_squared_error(y_val_u, pred_val_u, squared=False))

print("Official TEST MAE/RMSE:",
      mean_absolute_error(y_test_u, pred_test_u),
      mean_squared_error(y_test_u, pred_test_u, squared=False))


Selected features: 90
Official VAL  MAE/RMSE: 0.5680064871179141 0.7477074936313237
Official TEST MAE/RMSE: 0.594629198255745 0.7799127915124773


In [14]:
# Scaffold CV (separate experiment; full dataset)
X_all_2d = pd.concat([X_desc_train, X_desc_val, X_desc_test], axis=0)
y_all_2d = np.concatenate([y_train, y_val, y_test])
smiles_all_2d = np.concatenate([
    mordred_df_2d_train["smiles"].values,
    mordred_df_2d_val["smiles"].values,
    mordred_df_2d_test["smiles"].values
])

maes, rmses, nfeats = scaffold_cv_fast_selector_rf(X_all_2d, y_all_2d, smiles_all_2d)


Fold 1: MAE=0.5491 RMSE=0.7166 n_feat=120
Fold 2: MAE=0.5904 RMSE=0.7540 n_feat=90
Fold 3: MAE=0.5377 RMSE=0.7039 n_feat=50
Fold 4: MAE=0.5850 RMSE=0.7422 n_feat=120
Fold 5: MAE=0.5477 RMSE=0.7049 n_feat=210

5-fold scaffold CV:
MAE  mean±std: 0.5620 ± 0.0214
RMSE mean±std: 0.7243 ± 0.0203
n_feat mean±std: 118.0 ± 52.7


In [15]:
# Official MolNet benchmark (no leakage: train-only preprocessing)
X_desc_train = mordred_df_3d_train.drop(columns=["smiles", "target"])
y_train = mordred_df_3d_train["target"].values
smiles_train = mordred_df_3d_train["smiles"].values

X_desc_val = mordred_df_3d_val.drop(columns=["smiles", "target"])
y_val = mordred_df_3d_val["target"].values

X_desc_test = mordred_df_3d_test.drop(columns=["smiles", "target"])
y_test = mordred_df_3d_test["target"].values

groups_train = np.array([murcko_scaffold(s) for s in smiles_train])

Xtr_sel, Xva_sel, Xte_sel, info, rfecv_obj = fit_fast_selector_and_transform(
    X_desc_train, y_train, groups_train,
    X_desc_val, X_desc_test,
    vt_threshold=1e-8,
    corr_threshold=0.90,
    top_k=300,
    min_features=50,
    rfecv_cv_splits=5,
    rfecv_step=0.1
)

print("Selected features:", len(info["kept_final"]))

final_rf = RandomForestRegressor(
    n_estimators=1500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
final_rf.fit(Xtr_sel, y_train)

pred_val = final_rf.predict(Xva_sel)
pred_test = final_rf.predict(Xte_sel)

y_val_u = untransform_y(y_val, transformers)
pred_val_u = untransform_y(pred_val, transformers)
y_test_u = untransform_y(y_test, transformers)
pred_test_u = untransform_y(pred_test, transformers)

print("Official VAL  MAE/RMSE:",
      mean_absolute_error(y_val_u, pred_val_u),
      mean_squared_error(y_val_u, pred_val_u, squared=False))

print("Official TEST MAE/RMSE:",
      mean_absolute_error(y_test_u, pred_test_u),
      mean_squared_error(y_test_u, pred_test_u, squared=False))


Selected features: 90
Official VAL  MAE/RMSE: 0.5684409606456665 0.7391895460616532
Official TEST MAE/RMSE: 0.5953594817709555 0.7809425324565732


In [16]:
# Scaffold CV (separate experiment; full dataset)
X_all_3d = pd.concat([X_desc_train, X_desc_val, X_desc_test], axis=0)
y_all_3d = np.concatenate([y_train, y_val, y_test])
smiles_all_3d = np.concatenate([
    mordred_df_3d_train["smiles"].values,
    mordred_df_3d_val["smiles"].values,
    mordred_df_3d_test["smiles"].values
])

maes, rmses, nfeats = scaffold_cv_fast_selector_rf(X_all_3d, y_all_3d, smiles_all_3d)


Fold 1: MAE=0.5556 RMSE=0.7249 n_feat=210
Fold 2: MAE=0.6007 RMSE=0.7706 n_feat=210
Fold 3: MAE=0.5545 RMSE=0.7059 n_feat=150
Fold 4: MAE=0.5776 RMSE=0.7638 n_feat=50
Fold 5: MAE=0.5445 RMSE=0.6923 n_feat=180

5-fold scaffold CV:
MAE  mean±std: 0.5666 ± 0.0202
RMSE mean±std: 0.7315 ± 0.0310
n_feat mean±std: 160.0 ± 59.3


In [17]:
extra_3d_cols = sorted(set(X_all_3d.columns) - set(X_all_2d.columns))

selected_3d = set(info["kept_final"])  # or whatever you store
picked_extra = sorted(selected_3d.intersection(extra_3d_cols))
len(picked_extra), picked_extra[:20]

(5, ['FNSA3', 'FNSA5', 'FPSA4', 'PNSA1', 'RPSA'])

## Molecular Fingerprints

In [18]:
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs.cDataStructs import ConvertToNumpyArray

def maccs_to_numpy(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr

maccs_train = data_train['smiles'].apply(maccs_to_numpy)
maccs_val = data_val['smiles'].apply(maccs_to_numpy)
maccs_test = data_test['smiles'].apply(maccs_to_numpy)

In [19]:
from rdkit.Chem import rdFingerprintGenerator

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)
ecfp_train = data_train['smiles'].apply(
    lambda x: np.array(list(morgan_gen.GetFingerprint(Chem.MolFromSmiles(x)).ToBitString()), dtype=int))
ecfp_val = data_val['smiles'].apply(
    lambda x: np.array(list(morgan_gen.GetFingerprint(Chem.MolFromSmiles(x)).ToBitString()), dtype=int))
ecfp_test = data_test['smiles'].apply(
    lambda x: np.array(list(morgan_gen.GetFingerprint(Chem.MolFromSmiles(x)).ToBitString()), dtype=int))

In [20]:
from rdkit.Chem.Pharm2D import Generate, Gobbi_Pharm2D

factory = Gobbi_Pharm2D.factory

def erg_to_dense(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = Generate.Gen2DFingerprint(mol, factory)  # SparseBitVect

    n_bits = fp.GetNumBits()
    arr = np.zeros(n_bits, dtype=np.float32)

    for idx in fp.GetOnBits():
        arr[idx] = 1.0

    return arr


erg_train = np.vstack([erg_to_dense(s) for s in data_train["smiles"].values])
erg_val   = np.vstack([erg_to_dense(s) for s in data_val["smiles"].values])
erg_test  = np.vstack([erg_to_dense(s) for s in data_test["smiles"].values])


In [21]:
print(len(maccs_train.tolist()[0]), len(ecfp_train.tolist()[0]), len(erg_train.tolist()[0]))

167 512 39972


In [22]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD

def make_sgdr(alpha=1e-4):
    return SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=alpha,
        max_iter=5000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
    )


def make_erg_svd_sgdr(n_components=256, alpha=1e-4):
    return Pipeline([
        ("svd", TruncatedSVD(n_components=n_components, random_state=42)),
        ("sgd", make_sgdr(alpha=alpha)),
    ])


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def train_and_evaluate_model_fp(X_train, y_train, X_val, y_val, X_test, y_test, erg=False):

    if not erg:
        model = make_sgdr(alpha=1e-4)         # for MACCS/ECFP
    else:
        model = make_erg_svd_sgdr()  # for ErG
    # model = make_erg_svd_sgdr(256, 1e-4) # for ErG

    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)

    y_val_u = untransform_y(y_val, transformers)
    y_test_u = untransform_y(y_test, transformers)
    pred_val_u = untransform_y(pred_val, transformers)
    pred_test_u = untransform_y(pred_test, transformers)

    print("VAL  MAE/RMSE:",
        mean_absolute_error(y_val_u, pred_val_u),
        mean_squared_error(y_val_u, pred_val_u, squared=False))
    print("TEST MAE/RMSE:",
        mean_absolute_error(y_test_u, pred_test_u),
        mean_squared_error(y_test_u, pred_test_u, squared=False))


In [24]:
print('MACCS')
train_and_evaluate_model_fp(
    np.vstack(maccs_train.values), data_train["target"].values, 
    np.vstack(maccs_val.values), data_val["target"].values,
    np.vstack(maccs_test.values), data_test["target"].values)

print('ECFP')
train_and_evaluate_model_fp(
    np.vstack(ecfp_train.values), data_train["target"].values,
    np.vstack(ecfp_val.values), data_val["target"].values,
    np.vstack(ecfp_test.values), data_test["target"].values)

print('ErG')
train_and_evaluate_model_fp(
    erg_train, data_train["target"].values,
    erg_val, data_val["target"].values,
    erg_test, data_test["target"].values, erg=True)

MACCS
VAL  MAE/RMSE: 0.8223822076198779 1.0265900870606617
TEST MAE/RMSE: 0.7963019951369045 1.0030147925447503
ECFP
VAL  MAE/RMSE: 0.8623844753035548 1.1072604293739337
TEST MAE/RMSE: 0.8291889079289632 1.0603924576489252
ErG
VAL  MAE/RMSE: 0.8114760222330994 1.0285271174784898
TEST MAE/RMSE: 0.7482906074054965 0.9364542805590843


In [25]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

def scaffold_cv_regression(X, y, groups, make_model, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    maes, rmses = [], []

    for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
        model = make_model()
        model.fit(X[tr], y[tr])
        pred = model.predict(X[va])

        y_va_u = untransform_y(y[va], transformers)
        pred_u = untransform_y(pred, transformers)

        mae = mean_absolute_error(y_va_u, pred_u)
        rmse = mean_squared_error(y_va_u, pred_u, squared=False)
        maes.append(mae); rmses.append(rmse)

        print(f"Fold {fold}: MAE={mae:.4f} RMSE={rmse:.4f}")

    print("5-fold scaffold CV:")
    print(f"MAE  mean+/-std: {np.mean(maes):.4f} +/- {np.std(maes):.4f}")
    print(f"RMSE mean+/-std: {np.mean(rmses):.4f} +/- {np.std(rmses):.4f}")
    return maes, rmses


In [26]:
# Scaffold CV (separate experiment; full dataset)
groups_all = np.array([murcko_scaffold(s) for s in smiles_all])

maccs_all = np.vstack([
    np.vstack(maccs_train.values),
    np.vstack(maccs_val.values),
    np.vstack(maccs_test.values)
])
ecfp_all = np.vstack([
    np.vstack(ecfp_train.values),
    np.vstack(ecfp_val.values),
    np.vstack(ecfp_test.values)
])
erg_all = np.vstack([erg_train, erg_val, erg_test])

print('MACCS')
scaffold_cv_regression(maccs_all, y_all, groups_all,
                       make_model=lambda: make_sgdr(1e-4))

print('ECFP')
scaffold_cv_regression(ecfp_all, y_all, groups_all,
                       make_model=lambda: make_sgdr(1e-4))

print('ErG')
scaffold_cv_regression(erg_all, y_all, groups_all,
                       make_model=lambda: make_erg_svd_sgdr(256, 1e-4))


MACCS
Fold 1: MAE=0.8037 RMSE=0.9948
Fold 2: MAE=0.7931 RMSE=1.0138
Fold 3: MAE=0.7876 RMSE=0.9869
Fold 4: MAE=0.7173 RMSE=0.8980
Fold 5: MAE=0.8156 RMSE=1.0244

5-fold scaffold CV:
MAE  mean±std: 0.7834 ± 0.0344
RMSE mean±std: 0.9836 ± 0.0448
ECFP
Fold 1: MAE=0.8309 RMSE=1.0505
Fold 2: MAE=0.8731 RMSE=1.1097
Fold 3: MAE=0.7897 RMSE=1.0002
Fold 4: MAE=0.7609 RMSE=0.9679
Fold 5: MAE=0.8472 RMSE=1.0537

5-fold scaffold CV:
MAE  mean±std: 0.8204 ± 0.0402
RMSE mean±std: 1.0364 ± 0.0487
ErG
Fold 1: MAE=0.7893 RMSE=1.0134
Fold 2: MAE=0.8141 RMSE=1.0230
Fold 3: MAE=0.7426 RMSE=0.9179
Fold 4: MAE=0.6794 RMSE=0.8637
Fold 5: MAE=0.7571 RMSE=0.9777

5-fold scaffold CV:
MAE  mean±std: 0.7565 ± 0.0459
RMSE mean±std: 0.9591 ± 0.0603


([0.7893350026820796,
  0.8141029448610824,
  0.7425654680925755,
  0.6794215140989112,
  0.7570804311856023],
 [1.013381942607765,
  1.0229508083936405,
  0.9179406282163042,
  0.8636639528895754,
  0.9777340554942274])

## Mol2Vec

In [27]:
from tqdm import tqdm

# zbog nekompatibilnosti verzija
def sentences2vec(sentences, model, unseen='UNK'):
    """
    Convert a list of sentences (molecular substructure tokens) into vector representations
    using a gensim Word2Vec model (Gensim 4.x compatible).

    Parameters:
    - sentences: list of lists of substructure tokens (e.g., [['[C@@H]', '[CH3]'], ...])
    - model: gensim Word2Vec model (must be trained or preloaded)
    - unseen: token to use for unseen words (default 'UNK')

    Returns:
    - list of numpy arrays (each array = vector for one molecule)
    """
    vectors = []
    for sentence in tqdm(sentences, desc="Embedding molecules"):
        vecs = []
        for word in sentence:
            if word in model.wv:
                vecs.append(model.wv[word])
            elif unseen in model.wv:
                vecs.append(model.wv[unseen])
        if vecs:
            vectors.append(np.mean(vecs, axis=0))
        else:
            # In case no known words in sentence, use zero vector
            vectors.append(np.zeros(model.vector_size))
    return vectors

In [28]:
import random
from pathlib import Path
import gensim
from gensim.models import Word2Vec
from mol2vec.features import mol2alt_sentence
# from tqdm import tqdm
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# Train-only Mol2Vec to avoid spillage
MOL2VEC_MODEL_PATH = Path("./models/mol2vec_train_only_300dim.pkl")
RETRAIN_MOL2VEC = False
MOL2VEC_VECTOR_SIZE = 300

sentences_train = [mol2alt_sentence(Chem.MolFromSmiles(smi), radius=1) for smi in data_train['smiles'].tolist()]

if RETRAIN_MOL2VEC or not MOL2VEC_MODEL_PATH.exists():
    random.seed(42)
    np.random.seed(42)
    model = Word2Vec(
        sentences=sentences_train,
        vector_size=MOL2VEC_VECTOR_SIZE,
        window=10,
        min_count=1,
        workers=1,
        sg=1,
        seed=42,
        epochs=20
    )
    MOL2VEC_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(MOL2VEC_MODEL_PATH))
else:
    model = Word2Vec.load(str(MOL2VEC_MODEL_PATH))

sentences_val = [mol2alt_sentence(Chem.MolFromSmiles(smi), radius=1) for smi in data_val['smiles'].tolist()]
sentences_test = [mol2alt_sentence(Chem.MolFromSmiles(smi), radius=1) for smi in data_test['smiles'].tolist()]

# Convert to vectors
vectors_train = np.array(sentences2vec(sentences_train, model))
vectors_val = np.array(sentences2vec(sentences_val, model))
vectors_test = np.array(sentences2vec(sentences_test, model))


In [29]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

svr = make_pipeline(
    StandardScaler(),
    SVR(C=10.0, gamma="scale", epsilon=0.1, kernel="rbf")
)

X_train_m2v = np.vstack(vectors_train)
y_train = data_train["target"].values
X_val_m2v = np.vstack(vectors_val)
y_val = data_val["target"].values
X_test_m2v = np.vstack(vectors_test)
y_test = data_test["target"].values

svr.fit(X_train_m2v, y_train)
pred_val = svr.predict(X_val_m2v)
pred_test = svr.predict(X_test_m2v)

for transformer in transformers:
    if transformer.transform_y:
        y_val_u = transformer.untransform(y_val.reshape(-1, 1)).reshape(-1)
        pred_val_u = transformer.untransform(pred_val.reshape(-1, 1)).reshape(-1)
        y_test_u = transformer.untransform(y_test.reshape(-1, 1)).reshape(-1)
        pred_test_u = transformer.untransform(pred_test.reshape(-1, 1)).reshape(-1)


print("Official VAL  MAE/RMSE:",
      mean_absolute_error(y_val_u, pred_val_u),
      mean_squared_error(y_val_u, pred_val_u, squared=False))

print("Official TEST MAE/RMSE:",
      mean_absolute_error(y_test_u, pred_test_u),
      mean_squared_error(y_test_u, pred_test_u, squared=False))


Official VAL  MAE/RMSE: 0.5742155955923045 0.740675647689356
Official TEST MAE/RMSE: 0.6067213188969071 0.7765996193819946


In [30]:
from sklearn.model_selection import GroupKFold

def scaffold_cv_on_precomputed_X(X_all, y_all, smiles_all, transformers=None, n_splits=5):
    groups = np.array([murcko_scaffold(s) for s in smiles_all])
    cv = GroupKFold(n_splits=n_splits)

    maes, rmses = [], []
    for fold, (tr, va) in enumerate(cv.split(X_all, y_all, groups=groups), 1):
        model = make_pipeline(StandardScaler(), SVR(C=10.0, gamma="scale", epsilon=0.1, kernel="rbf"))
        model.fit(X_all[tr], y_all[tr])
        pred = model.predict(X_all[va])

        y_va_u = untransform_y(y_all[va], transformers)
        pred_u = untransform_y(pred, transformers)

        maes.append(mean_absolute_error(y_va_u, pred_u))
        rmses.append(mean_squared_error(y_va_u, pred_u, squared=False))
        print(f"Fold {fold}: MAE={maes[-1]:.4f}, RMSE={rmses[-1]:.4f}")

    print(f"5-fold scaffold-CV MAE  mean+/-std: {np.mean(maes):.4f} +/- {np.std(maes):.4f}")
    print(f"5-fold scaffold-CV RMSE mean+/-std: {np.mean(rmses):.4f} +/- {np.std(rmses):.4f}")
    return maes, rmses


In [31]:
# Scaffold CV (separate experiment; full dataset)
maes_m2v, rmses_m2v = scaffold_cv_on_precomputed_X(
    np.vstack([vectors_train, vectors_val, vectors_test]),
    y_all,
    smiles_all,
    transformers=transformers)


Fold 1: MAE=0.5627, RMSE=0.7516
Fold 2: MAE=0.5703, RMSE=0.7483
Fold 3: MAE=0.5592, RMSE=0.7289
Fold 4: MAE=0.5389, RMSE=0.7237
Fold 5: MAE=0.5800, RMSE=0.7671

5-fold scaffold-CV MAE  mean±std: 0.5622 ± 0.0137
5-fold scaffold-CV RMSE mean±std: 0.7439 ± 0.0158


# ChemBERTa

In [32]:
import torch
from transformers import AutoTokenizer, AutoModel

def mean_pool(last_hidden_state, attention_mask):
    # last_hidden_state: (B, L, H), attention_mask: (B, L)
    mask = attention_mask.unsqueeze(-1).float()  # (B, L, 1)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts  # (B, H)

@torch.no_grad()
def chemberta_embeddings(smiles_list, model_name="seyonec/ChemBERTa-zinc-base-v1",
                         batch_size=32, max_length=256, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # ✅ ensure list[str] (not numpy array)
    if not isinstance(smiles_list, list):
        smiles_list = smiles_list.tolist()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_vecs = []
    for i in range(0, len(smiles_list), batch_size):
        # ✅ ensure python list
        batch = smiles_list[i:i+batch_size]  # now it's a list slice

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        all_vecs.append(pooled.cpu().numpy().astype(np.float32))

    return np.vstack(all_vecs)


In [33]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

svr = make_pipeline(
    StandardScaler(),
    SVR(C=10.0, gamma="scale", epsilon=0.1, kernel="rbf")
)

In [34]:
from pathlib import Path

# SMILES + y from your already-defined official split dfs
smiles_train = data_train["smiles"].values
smiles_val   = data_val["smiles"].values
smiles_test  = data_test["smiles"].values

y_train = data_train["target"].values
y_val   = data_val["target"].values
y_test  = data_test["target"].values

# embeddings (train-only ChemBERTa)
CHEMBERTA_MODEL_DIR = Path("./chemberta_mlm_final")
if not CHEMBERTA_MODEL_DIR.exists():
    raise ValueError(
        "Train-only ChemBERTa model not found. Run the 'ChemBERTa training' section "
        "to create ./chemberta_mlm_final before computing embeddings."
    )

X_train_emb = chemberta_embeddings(smiles_train, model_name=str(CHEMBERTA_MODEL_DIR))
X_val_emb   = chemberta_embeddings(smiles_val,   model_name=str(CHEMBERTA_MODEL_DIR))
X_test_emb  = chemberta_embeddings(smiles_test,  model_name=str(CHEMBERTA_MODEL_DIR))

# (optional) reduction
# pca = PCA(n_components=256, random_state=42)
# X_train_emb = pca.fit_transform(X_train_emb)
# X_val_emb   = pca.transform(X_val_emb)
# X_test_emb  = pca.transform(X_test_emb)

svr.fit(X_train_emb, y_train)
pred_val  = svr.predict(X_val_emb)
pred_test = svr.predict(X_test_emb)

# untransform if needed (same as you do now)
y_val_u, pred_val_u = y_val, pred_val
y_test_u, pred_test_u = y_test, pred_test
for t in transformers:
    if getattr(t, "transform_y", False):
        y_val_u = t.untransform(y_val_u.reshape(-1,1)).ravel()
        pred_val_u = t.untransform(pred_val_u.reshape(-1,1)).ravel()
        y_test_u = t.untransform(y_test_u.reshape(-1,1)).ravel()
        pred_test_u = t.untransform(pred_test_u.reshape(-1,1)).ravel()

from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Official VAL  MAE/RMSE:",
      mean_absolute_error(y_val_u, pred_val_u),
      mean_squared_error(y_val_u, pred_val_u, squared=False))
print("Official TEST MAE/RMSE:",
      mean_absolute_error(y_test_u, pred_test_u),
      mean_squared_error(y_test_u, pred_test_u, squared=False))


Official VAL  MAE/RMSE: 0.7432221534542978 0.977353117560079
Official TEST MAE/RMSE: 0.6675005309016272 0.8549974263956673


In [35]:
from sklearn.model_selection import GroupKFold
from rdkit.Chem.Scaffolds import MurckoScaffold

def murcko_scaffold(smiles: str) -> str:
    return MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles, includeChirality=False)

def scaffold_cv_on_precomputed_X(X_all, y_all, smiles_all, transformers=None, n_splits=5):
    groups = np.array([murcko_scaffold(s) for s in smiles_all])
    cv = GroupKFold(n_splits=n_splits)

    maes, rmses = [], []
    for fold, (tr, va) in enumerate(cv.split(X_all, y_all, groups=groups), 1):
        model = make_pipeline(StandardScaler(), SVR(C=10.0, gamma="scale", epsilon=0.1))
        model.fit(X_all[tr], y_all[tr])
        pred = model.predict(X_all[va])

        y_va_u = untransform_y(y_all[va], transformers)
        pred_u = untransform_y(pred, transformers)

        maes.append(mean_absolute_error(y_va_u, pred_u))
        rmses.append(mean_squared_error(y_va_u, pred_u, squared=False))
        print(f"Fold {fold}: MAE={maes[-1]:.4f}, RMSE={rmses[-1]:.4f}")

    print(f"5-fold scaffold-CV MAE  mean+/-std: {np.mean(maes):.4f} +/- {np.std(maes):.4f}")
    print(f"5-fold scaffold-CV RMSE mean+/-std: {np.mean(rmses):.4f} +/- {np.std(rmses):.4f}")
    return maes, rmses


In [36]:
# Scaffold CV (separate experiment; full dataset)
X_all_cb = np.vstack([X_train_emb, X_val_emb, X_test_emb])

maes_cb, rmses_cb = scaffold_cv_on_precomputed_X(X_all_cb, y_all, smiles_all, transformers=transformers)


Fold 1: MAE=0.7310, RMSE=0.9421
Fold 2: MAE=0.7707, RMSE=0.9756
Fold 3: MAE=0.6848, RMSE=0.8718
Fold 4: MAE=0.6712, RMSE=0.8623
Fold 5: MAE=0.6937, RMSE=0.9052

5-fold scaffold-CV MAE  mean±std: 0.7103 ± 0.0361
5-fold scaffold-CV RMSE mean±std: 0.9114 ± 0.0426


### ChemBERTa training

In [37]:
# !pip install -U "accelerate>=0.26.0"

In [38]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# 1) Load SMILES corpus (list of strings)
# Example: only from training split (recommended for strictness)
smiles_corpus = data_train["smiles"].astype(str).tolist()

# Optional: add extra SMILES from an external file (not recommended for strict no-spillage)
# with open("external_smiles.txt") as f:
#     smiles_corpus += [line.strip() for line in f if line.strip()]

# 2) Tokenizer + MLM model
# NOTE: Set INIT_FROM_BASE=False for strict no-spillage (random init)
INIT_FROM_BASE = False
model_name = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if INIT_FROM_BASE:
    model = AutoModelForMaskedLM.from_pretrained(model_name)
else:
    config = AutoConfig.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_config(config)

# Optional: memory savers
model.gradient_checkpointing_enable()  # reduces VRAM, increases compute a bit

# 3) Dataset -> tokenized
ds = Dataset.from_dict({"text": smiles_corpus})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,   # 128 if VRAM tight; 256 is usually ok for SMILES
    )

ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])

# 4) MLM collator
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# 5) Training args (laptop GPU-friendly defaults)
args = TrainingArguments(
    output_dir="./chemberta_mlm_ckpts",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,          # tune down if OOM
    gradient_accumulation_steps=4,          # effective batch=32
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),         # mixed precision if GPU
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok,
    data_collator=collator,
)

trainer.train()

# 6) Save final model + tokenizer
final_dir = "./chemberta_mlm_final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved to:", final_dir)


Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

Step,Training Loss
50,0.845500
100,0.600900
150,0.544800
200,0.525800
250,0.485100
300,0.504800


Saved to: ./chemberta_mlm_final


In [39]:
X_train_emb_cb = chemberta_embeddings(smiles_train, model_name="./chemberta_mlm_final")
X_val_emb_cb = chemberta_embeddings(smiles_val,   model_name="./chemberta_mlm_final")
X_test_emb_cb = chemberta_embeddings(smiles_test,  model_name="./chemberta_mlm_final")

In [40]:
svr.fit(X_train_emb_cb, y_train)
pred_val  = svr.predict(X_val_emb_cb)
pred_test = svr.predict(X_test_emb_cb)

# untransform if needed (same as you do now)
y_val_u, pred_val_u = y_val, pred_val
y_test_u, pred_test_u = y_test, pred_test
for t in transformers:
    if getattr(t, "transform_y", False):
        y_val_u = t.untransform(y_val_u.reshape(-1,1)).ravel()
        pred_val_u = t.untransform(pred_val_u.reshape(-1,1)).ravel()
        y_test_u = t.untransform(y_test_u.reshape(-1,1)).ravel()
        pred_test_u = t.untransform(pred_test_u.reshape(-1,1)).ravel()

from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Official VAL  MAE/RMSE:",
      mean_absolute_error(y_val_u, pred_val_u),
      mean_squared_error(y_val_u, pred_val_u, squared=False))
print("Official TEST MAE/RMSE:",
      mean_absolute_error(y_test_u, pred_test_u),
      mean_squared_error(y_test_u, pred_test_u, squared=False))


Official VAL  MAE/RMSE: 0.7374795208745977 0.9607695873771006
Official TEST MAE/RMSE: 0.6838374732712966 0.8851641331309503


In [41]:
# Scaffold CV (separate experiment; full dataset)
X_all_cb1 = np.vstack([X_train_emb_cb, X_val_emb_cb, X_test_emb_cb])

maes_cb1, rmses_cb1 = scaffold_cv_on_precomputed_X(X_all_cb1, y_all, smiles_all, transformers=transformers)


Fold 1: MAE=0.7169, RMSE=0.9391
Fold 2: MAE=0.7490, RMSE=0.9618
Fold 3: MAE=0.6580, RMSE=0.8376
Fold 4: MAE=0.6616, RMSE=0.8467
Fold 5: MAE=0.7287, RMSE=0.9409

5-fold scaffold-CV MAE  mean±std: 0.7028 ± 0.0367
5-fold scaffold-CV RMSE mean±std: 0.9052 ± 0.0522
